In [1]:
import numpy as np
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from PIL import Image
import joblib
from pathlib import Path
import gc

In [2]:
train_dir = "./data/train"
test_dir = "./data/test"
IMG_SIZE = 64

In [3]:
def load_images_from_directory(data_dir, img_size):
    """Load images and labels from directory structure."""
    images = []
    labels = []
    class_names = sorted([d for d in os.listdir(data_dir) 
                         if os.path.isdir(os.path.join(data_dir, d))])
    
    print(f"\nLoading images from {data_dir}")
    print(f"Classes found: {class_names}")
    
    for label_idx, class_name in enumerate(class_names):
        class_dir = os.path.join(data_dir, class_name)
        image_files = list(Path(class_dir).glob('*.jpg')) + \
                     list(Path(class_dir).glob('*.jpeg')) + \
                     list(Path(class_dir).glob('*.png'))
        
        print(f"Loading {class_name}: {len(image_files)} images...")
        
        for img_path in image_files:
            try:
                # Load and resize image
                img = Image.open(img_path).convert('RGB')
                img = img.resize((img_size, img_size))
                
                # Convert to numpy array and flatten
                img_array = np.array(img).flatten()
                
                images.append(img_array)
                labels.append(label_idx)
                
            except Exception as e:
                print(f"  Skipping {img_path.name}: {str(e)[:50]}")
                continue
        
        # Progress update
        print(f"  Loaded {len([l for l in labels if l == label_idx])} images")
    
    return np.array(images), np.array(labels), class_names

In [4]:
X_train, y_train, class_names = load_images_from_directory(train_dir, IMG_SIZE)
print(f"\nTraining data shape: {X_train.shape}")
print(f"Training labels shape: {y_train.shape}")


Loading images from ./data/train
Classes found: ['galaxy', 'nebula', 'planet', 'star', 'unknown']
Loading galaxy: 28512 images...
  Loaded 28512 images
Loading nebula: 1221 images...
  Loaded 1221 images
Loading planet: 60801 images...
  Loaded 60801 images
Loading star: 36 images...
  Loaded 36 images
Loading unknown: 1689 images...
  Loaded 1689 images

Training data shape: (92259, 12288)
Training labels shape: (92259,)


In [5]:
X_test, y_test, _ = load_images_from_directory(test_dir, IMG_SIZE)
print(f"\nTest data shape: {X_test.shape}")
print(f"Test labels shape: {y_test.shape}")


Loading images from ./data/test
Classes found: ['galaxy', 'nebula', 'planet', 'star', 'unknown']
Loading galaxy: 108 images...
  Loaded 108 images
Loading nebula: 1567 images...
  Loaded 1567 images
Loading planet: 643 images...
  Loaded 643 images
Loading star: 17 images...
  Loaded 17 images
Loading unknown: 347 images...
  Loaded 347 images

Test data shape: (2682, 12288)
Test labels shape: (2682,)


In [6]:
X_train = X_train / 255.0
X_test = X_test / 255.0
print(f"\nData normalized to range [{X_train.min():.2f}, {X_train.max():.2f}]")


Data normalized to range [0.00, 1.00]


In [7]:
print("\nTraining Random Forest Classifier...")
rf_model = RandomForestClassifier(
    n_estimators=100,        # Number of trees
    max_depth=8,            # Maximum depth of trees
    min_samples_split=5,     # Minimum samples to split a node
    min_samples_leaf=2,      # Minimum samples at leaf node
    max_features='sqrt',     # Number of features to consider at each split
    n_jobs=-1,               # Use all CPU cores
    random_state=42,
    verbose=1
)

rf_model.fit(X_train, y_train)
print("\nTraining complete!")


Training Random Forest Classifier...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:  1.2min
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:  3.5min finished



Training complete!


In [8]:
y_pred = rf_model.predict(X_test)

[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.1s finished


In [9]:
test_acc = accuracy_score(y_test, y_pred)
print(f"\nTest Accuracy: {round(test_acc * 100, 2)}%")


Test Accuracy: 91.61%


In [10]:
print("\nModel Performance Summary:")
print(f"Overall Accuracy: {test_acc * 100:.2f}%")
print(f"Total test samples: {len(y_test)}")

for i, class_name in enumerate(class_names):
    class_mask = y_test == i
    num_samples = class_mask.sum()
    
    if num_samples > 0:
        correct = ((y_test == i) & (y_pred == i)).sum()
        accuracy = correct / num_samples * 100
        
        print(f"\n{class_name}:")
        print(f"  Samples: {num_samples}")
        print(f"  Correct: {correct}")
        print(f"  Accuracy: {accuracy:.2f}%")


Model Performance Summary:
Overall Accuracy: 91.61%
Total test samples: 2682

galaxy:
  Samples: 108
  Correct: 108
  Accuracy: 100.00%

nebula:
  Samples: 1567
  Correct: 1391
  Accuracy: 88.77%

planet:
  Samples: 643
  Correct: 643
  Accuracy: 100.00%

star:
  Samples: 17
  Correct: 0
  Accuracy: 0.00%

unknown:
  Samples: 347
  Correct: 315
  Accuracy: 90.78%


In [11]:
print("\nCalculating feature importance...")
feature_importance = rf_model.feature_importances_
print(f"Feature importance calculated (top 10 pixels):")
top_features = np.argsort(feature_importance)[-10:][::-1]
for idx in top_features:
    print(f"  Pixel {idx}: {feature_importance[idx]:.6f}")

gc.collect()


Calculating feature importance...
Feature importance calculated (top 10 pixels):
  Pixel 665: 0.034041
  Pixel 674: 0.025584
  Pixel 11234: 0.025410
  Pixel 1247: 0.025347
  Pixel 872: 0.017732
  Pixel 1067: 0.017383
  Pixel 1043: 0.017301
  Pixel 1070: 0.017144
  Pixel 482: 0.017067
  Pixel 1061: 0.017058


69

In [12]:
os.makedirs("results", exist_ok=True)

with open("results/random_forest_performance.txt", "w") as f:
    f.write("RANDOM FOREST MODEL PERFORMANCE\n")
    f.write(f"Test Accuracy: {round(test_acc * 100, 2)}%\n\n")
    
    f.write("Classes:\n")
    for i, c in enumerate(class_names):
        f.write(f"  {i}: {c}\n")
    
    f.write("\n" + "=" * 60 + "\n")
    f.write("PER-CLASS PERFORMANCE\n")
    f.write("=" * 60 + "\n\n")
    
    for i, class_name in enumerate(class_names):
        class_mask = y_test == i
        if class_mask.sum() > 0:
            correct = ((y_test == i) & (y_pred == i)).sum()
            total = class_mask.sum()
            accuracy = correct / total * 100
            f.write(f"{class_name}:\n")
            f.write(f"  Samples: {total}\n")
            f.write(f"  Correct: {correct}\n")
            f.write(f"  Accuracy: {accuracy:.2f}%\n\n")
    
    f.write("MODEL CONFIGURATION\n")
    f.write(f"Image Size: {IMG_SIZE}x{IMG_SIZE}\n")
    f.write(f"Number of Trees: {rf_model.n_estimators}\n")
    f.write(f"Max Depth: {rf_model.max_depth}\n")
    f.write(f"Features per Split: {rf_model.max_features}\n")

print("Performance report saved to results/random_forest_performance.txt")

Performance report saved to results/random_forest_performance.txt


In [13]:
joblib.dump(rf_model, "results/random_forest_model.pkl")
print("Model saved to results/random_forest_model.pkl")

Model saved to results/random_forest_model.pkl


In [14]:
print("MODEL SUMMARY")
print(f"Model Type: Random Forest Classifier")
print(f"Number of Trees: {rf_model.n_estimators}")
print(f"Max Tree Depth: {rf_model.max_depth}")
print(f"Number of Classes: {len(class_names)}")
print(f"Input Features: {X_train.shape[1]} (flattened {IMG_SIZE}x{IMG_SIZE}x3 image)")
print(f"Training Samples: {len(X_train)}")
print(f"Test Samples: {len(X_test)}")

gc.collect()

MODEL SUMMARY
Model Type: Random Forest Classifier
Number of Trees: 100
Max Tree Depth: 8
Number of Classes: 5
Input Features: 12288 (flattened 64x64x3 image)
Training Samples: 92259
Test Samples: 2682


0